# Paper 2 · Pelengkap naskah (CPU)

Notebook ini **tidak butuh GPU**. Ia hanya membaca file hasil langkah 4, 6, dan 7 di Drive, lalu membuat tiga hal untuk naskah:

1. **Selang kepercayaan uji deletion pada kondisi simetris** (untuk kalimat kosong di bagian 3.5).
2. **Kesepakatan peta occlusion antar kondisi**: seberapa mirip peta untuk gambar yang sama saat hanya riwayat kompresinya yang berbeda.
3. **Figure 3**: peta occlusion untuk gambar yang sama pada tiga kondisi, ditambah lembar kandidat supaya contohnya bisa diganti.

**Sebelum Run all:** menu **Runtime → Change runtime type → CPU → Save**. Selesai dalam beberapa menit.

## 1. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import time, platform, shutil, zipfile
import numpy as np, pandas as pd, scipy
from pathlib import Path
from scipy import stats
import torch, torch.nn.functional as F
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
N_BOOT = 2000
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
OLD_STYLES = ['baroque', 'renaissance', 'romanticism', 'ukiyo_e']   # gaya yang karyanya umumnya sudah domain publik

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES, RESULTS, CACHE = PROJECT / 'catatan', PROJECT / 'hasil', PROJECT / 'data_olahan'
FIGS = RESULTS / 'gambar_naskah'; FIGS.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/work'); LOCAL.mkdir(parents=True, exist_ok=True)
for f in ('rq2_elemen_simetris_per_gambar.csv',):
    assert (RESULTS / f).exists(), f'File {f} belum ada. Jalankan notebook langkah 7 dulu.'
for folder in ('peta', 'peta_logit', 'peta_simetris'):
    assert len(list((RESULTS / folder).glob('*fold*.npz'))) >= 5, f'Folder hasil/{folder} belum lengkap.'

manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
manifest['y'] = (manifest['label'] == 'ai').astype(int)
env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\nnumpy: {np.__version__}\n'
       f'pandas: {pd.__version__}\nscipy: {scipy.__version__}\ntorch: {torch.__version__}\nmatplotlib: {matplotlib.__version__}\n'
       f'bootstrap: {N_BOOT} ulangan, seed {SEED}\n')
(NOTES / 'lingkungan_pelengkap_naskah.txt').write_text(env)
print(env)

## 2. Selang kepercayaan uji deletion, kondisi simetris

Untuk tiap gambar: **keunggulan** = luas kurva kontrol terbaik dikurangi luas kurva peta occlusion. Positif berarti peta lebih setia. Kontrol terbaik adalah yang rata-ratanya paling kecil di kelompok itu (tengah atau acak). Peta dinyatakan unggul kalau batas bawah selang 95% di atas nol.

In [ ]:
df = pd.read_csv(RESULTS / 'rq2_elemen_simetris_per_gambar.csv')
GROUPS = ['5-fold: manusia benar', '5-fold: LD benar', '5-fold: SD benar']
rng = np.random.default_rng(SEED)

def boot_ci(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; n = len(x)
    if n < 10: return np.nan, np.nan, np.nan, n
    m = x[rng.integers(0, n, size=(N_BOOT, n))].mean(1)
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5), n

rows = []
for grp in GROUPS:
    g = df[df.kelompok == grp]
    if len(g) == 0: continue
    for r in ('blur', 'mean'):
        ctrl = min(('centre', 'random'), key=lambda c: g[f'del_{r}_{c}'].mean())
        diff = g[f'del_{r}_{ctrl}'] - g[f'del_{r}_occ']
        mean, lo, hi, n = boot_ci(diff)
        rows.append({'kelompok': grp, 'pengganti': 'blur' if r == 'blur' else 'warna rata-rata', 'kontrol_terbaik': 'tengah' if ctrl == 'centre' else 'acak',
                     'n': n, 'luas_peta': g[f'del_{r}_occ'].mean(), 'luas_kontrol': g[f'del_{r}_{ctrl}'].mean(),
                     'keunggulan': mean, 'ci_bawah': lo, 'ci_atas': hi, 'unggul': bool(lo > 0) if not np.isnan(lo) else False,
                     'persen_gambar_peta_lebih_baik': 100 * float((diff > 0).mean())})
B = pd.DataFrame(rows); B.to_csv(RESULTS / 'rq2_simetris_deletion_selang_kepercayaan.csv', index=False)
print('B. KEUNGGULAN PETA OCCLUSION ATAS KONTROL TERBAIK, kondisi simetris (selang bootstrap 95%)')
print(B.round(3).to_string(index=False))

## 3. Kesepakatan peta antar kondisi

Untuk gambar yang sama, peta occlusion dari tiga kondisi dibandingkan berpasangan: korelasi Spearman antar peta, dan tumpang tindih (IoU) area sorotan 20%. Dua area acak seluas 20% punya IoU sekitar 0,11. Hanya gambar yang ditebak benar di kedua kondisi dan petanya tidak kosong yang dihitung.

Catatan: peta kondisi normal memakai skala probabilitas (langkah 4); dua kondisi lain memakai skala logit.

In [ ]:
def patch_to_cells(d):
    k, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C); cnt = torch.zeros(C, C)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def load_condition(cond):
    ids, preds, cells = [], [], []
    for k in range(5):
        if cond == 'normal':
            z = np.load(RESULTS / 'peta' / f'normal_fold{k}.npz', allow_pickle=True)
            pred = z['pred'].astype(int); occ = z['occ'][np.arange(len(pred)), pred]
        elif cond == 'satu_sisi':
            z = np.load(RESULTS / 'peta_logit' / f'riwayat_disamakan_fold{k}.npz', allow_pickle=True)
            pred = z['pred'].astype(int); occ = z['occ_logit']
        else:
            z = np.load(RESULTS / 'peta_simetris' / f'simetris_fold{k}.npz', allow_pickle=True)
            pred = z['pred'].astype(int); occ = z['occ_logit']
        ids.append(z['sample_id'].astype(str)); preds.append(pred)
        cells.append(patch_to_cells(torch.from_numpy(np.asarray(occ, dtype=np.float32))).clamp(min=0))
    ids = np.concatenate(ids); order = np.argsort(ids)
    return ids[order], np.concatenate(preds)[order], torch.cat(cells)[order]

CONDS = ['normal', 'satu_sisi', 'simetris']
NAMA = {'normal': 'normal', 'satu_sisi': 'satu sisi', 'simetris': 'simetris'}
M = {c: load_condition(c) for c in CONDS}
base_ids = M['simetris'][0]
for c in CONDS: assert (M[c][0] == base_ids).all(), 'Urutan gambar antar kondisi tidak sama.'
info = manifest.set_index('sample_id').loc[base_ids]
y, src, style = info['y'].values, info['source'].values, info['style'].values
ok = {c: (M[c][1] == y) & (M[c][2].amax((1, 2)).numpy() > 0) for c in CONDS}

g = torch.Generator().manual_seed(SEED)
def region(cells):
    u = up(cells).reshape(len(cells), -1)
    u = u + 1e-7 * u.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(u.shape, generator=g)
    thr = u.kthvalue(int(round((1 - TOP) * u.shape[1])), dim=1, keepdim=True).values
    return (u > thr)
R = {c: region(M[c][2]) for c in CONDS}

def spearman_rows(a, b):
    if len(a) == 0: return np.array([])
    a = a.reshape(len(a), -1).numpy(); b = b.reshape(len(b), -1).numpy()
    return np.array([stats.spearmanr(a[i], b[i]).statistic for i in range(len(a))])

SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
PAIRS = [('normal', 'satu_sisi'), ('normal', 'simetris'), ('satu_sisi', 'simetris')]
per_image = pd.DataFrame({'sample_id': base_ids, 'source': src, 'style': style})
rows = []
for a, b in PAIRS:
    both = ok[a] & ok[b]
    rho = np.full(len(base_ids), np.nan); iou = np.full(len(base_ids), np.nan)
    idx = np.where(both)[0]
    rho[idx] = spearman_rows(M[a][2][idx], M[b][2][idx])
    if len(idx):
        inter = (R[a][idx] & R[b][idx]).sum(1).float(); union = (R[a][idx] | R[b][idx]).sum(1).float()
        iou[idx] = (inter / union).numpy()
    per_image[f'rho_{a}_{b}'] = rho; per_image[f'iou_{a}_{b}'] = iou
    for s in SRC:
        sel = both & (src == s)
        rows.append({'pasangan': f'{NAMA[a]} vs {NAMA[b]}', 'sumber': SRC[s], 'n': int(sel.sum()),
                     'median_korelasi': np.nanmedian(rho[sel]) if sel.any() else np.nan, 'median_iou': np.nanmedian(iou[sel]) if sel.any() else np.nan})
A = pd.DataFrame(rows); A.to_csv(RESULTS / 'rq2_kesepakatan_antar_kondisi.csv', index=False)
per_image.to_csv(RESULTS / 'rq2_kesepakatan_antar_kondisi_per_gambar.csv', index=False)
print('A. KESEPAKATAN PETA OCCLUSION ANTAR KONDISI (gambar yang sama; IoU acak sekitar 0,11)')
print(A.pivot(index='sumber', columns='pasangan', values=['median_korelasi', 'median_iou']).round(3).to_string())
print('\nJumlah gambar per sel:'); print(A.pivot(index='sumber', columns='pasangan', values='n').to_string())

## 4. Figure 3 dan lembar kandidat

**Aturan memilih contoh (supaya tidak memilih yang paling dramatis):** untuk tiap sumber, diambil gambar yang kesepakatan peta satu sisi vs simetris-nya paling dekat dengan **median** sumber itu, di antara gambar yang ditebak benar di ketiga kondisi. Jadi contohnya mewakili kasus tipikal.

**Lisensi:** contoh karya manusia hanya diambil dari gaya Baroque, Renaissance, Romanticism, dan Ukiyo-e. Nama file dicetak supaya status domain publiknya bisa dicek (pelukis wafat lebih dari 70 tahun lalu). Lembar kandidat memberi lima pilihan per sumber kalau contoh utama perlu diganti.

In [ ]:
local_raw = LOCAL / 'raw'
if not local_raw.exists():
    shutil.copy(CACHE / 'ai_artbench_sample_raw.zip', LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as zf: zf.extractall(local_raw)
def load_image(sample_id):
    r = manifest.set_index('sample_id').loc[sample_id]
    p = local_raw / r['source'] / r['style'] / Path(r['file_drive']).name
    with Image.open(p) as im:
        im = im.convert('RGB'); w, h = im.size
        if w != h:
            s = min(w, h); im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
        return np.asarray(im.resize((INPUT, INPUT), Image.BICUBIC)), Path(r['file_drive']).name

all_ok = ok['normal'] & ok['satu_sisi'] & ok['simetris']
key = per_image['rho_satu_sisi_simetris'].values
cands = {}
for s in SRC:
    sel = all_ok & (src == s)
    if s == 'human': sel = sel & np.isin(style, OLD_STYLES)
    idx = np.where(sel & ~np.isnan(key))[0]
    if len(idx) == 0:
        cands[s] = idx; print(f'{SRC[s]}: tidak ada gambar yang memenuhi syarat.'); continue
    med = np.nanmedian(key[all_ok & (src == s)])
    cands[s] = idx[np.argsort(np.abs(key[idx] - med))][:6]
    print(f'{SRC[s]}: median korelasi satu sisi vs simetris = {med:.3f}; kandidat (urut dari yang paling dekat median):')
    for j, i in enumerate(cands[s]):
        print(f'   {j + 1}. {base_ids[i]} | gaya {style[i]} | korelasi {key[i]:.3f} | file {load_image(base_ids[i])[1]}')

pos = {s: i for i, s in enumerate(base_ids)}
TITLES = ['Image', 'Normalized', 'One-sided', 'Symmetric']
def draw(sample_ids, labels, path, dpi=300):
    n = len(sample_ids)
    fig, axes = plt.subplots(n, 4, figsize=(6.3, 1.62 * n), dpi=dpi)
    axes = np.atleast_2d(axes)
    for r_, sid in enumerate(sample_ids):
        i = pos[sid]; img, _ = load_image(sid); gray = img.mean(2)
        axes[r_, 0].imshow(img); axes[r_, 0].set_ylabel(labels[r_], fontsize=7)
        for c_, cond in enumerate(CONDS):
            ax = axes[r_, c_ + 1]
            m = up(M[cond][2][i:i + 1])[0].numpy(); m = m / m.max() if m.max() > 0 else m
            ax.imshow(gray, cmap='gray', vmin=0, vmax=255)
            ax.imshow(m, cmap='magma', alpha=0.55, vmin=0, vmax=1)
            ax.contour(R[cond][i].reshape(INPUT, INPUT).numpy().astype(float), levels=[0.5], colors='white', linewidths=0.8)
        for c_ in range(4):
            axes[r_, c_].set_xticks([]); axes[r_, c_].set_yticks([])
            if r_ == 0: axes[r_, c_].set_title(TITLES[c_], fontsize=7.5)
    fig.subplots_adjust(wspace=0.03, hspace=0.05)
    fig.savefig(path, bbox_inches='tight', pad_inches=0.03); plt.show(); plt.close(fig)

# Figure 3: satu contoh per sumber. Untuk mengganti contoh, isi PILIH dengan sample_id dari daftar kandidat di atas.
PILIH = {'human': None, 'latent_diffusion': None, 'stable_diffusion': None}
LBL = {'human': 'Human', 'latent_diffusion': 'Latent Diffusion', 'stable_diffusion': 'Stable Diffusion'}
chosen = [str(PILIH[s] or base_ids[cands[s][0]]) for s in SRC if len(cands[s])]
labels = [LBL[s] for s in SRC if len(cands[s])]
print('\nFIGURE 3 memakai:', chosen)
draw(chosen, labels, FIGS / 'fig3_peta_tiga_kondisi.png')
pd.DataFrame({'sample_id': chosen, 'sumber': labels, 'file': [load_image(s)[1] for s in chosen],
              'gaya': [style[pos[s]] for s in chosen], 'korelasi_satu_sisi_simetris': [key[pos[s]] for s in chosen]}).to_csv(FIGS / 'fig3_daftar_gambar.csv', index=False)

# Lembar kandidat (resolusi rendah, hanya untuk memilih)
for s in SRC:
    if len(cands[s]) > 1:
        print(f'\nLEMBAR KANDIDAT: {SRC[s]} (urutan sama dengan daftar di atas)')
        draw([base_ids[i] for i in cands[s]], [str(j + 1) for j in range(len(cands[s]))], FIGS / f'kandidat_{SRC[s]}.png', dpi=110)
print('\nTersimpan di', FIGS)

## Selesai

Kalau tabel B, tabel A, Figure 3, dan tiga lembar kandidat muncul tanpa error, semua beres. Tekan **Cmd+S**, lalu kabari Claude.

Yang perlu kamu cek sendiri: nama file contoh **karya manusia** di Figure 3. Pastikan pelukisnya wafat lebih dari 70 tahun lalu. Kalau ragu, pilih kandidat lain dan isi `PILIH` di sel 4.